# 1. Introduction & Business Problem
## 💼 Beyond the Data: The Business of Weather & Air Quality

While weather and air quality are vital environmental metrics, predicting them accurately is a **multi-billion dollar commercial opportunity**. 

This notebook goes beyond standard Exploratory Data Analysis (EDA). The final output of this project is a **Feature-Engineered Data Product** a curated export of rule-based business triggers (like surge pricing multipliers and targeted marketing signals) designed to be plugged directly into BI dashboards like Tableau or PowerBI.

### 🎯 Key Commercial Applications Addressed:
* **🛒 Dynamic Marketing (Retail & HVAC):** Triggering targeted ads for air purifiers or ACs exactly when a heatwave or pollution spike hits.
* **📦 Supply Chain Optimization:** Helping retailers preemptively stock umbrellas or winter gear.
* **🏥 Healthcare Demand Forecasting:** Flagging high-risk health days to anticipate surges in ER visits.
* **🚗 Gig Economy Logistics:** Calculating a "Surge Multiplier" for food delivery and rideshare apps during poor weather or toxic air events.
* **⚡ Energy Grid Management:** Estimating power loads using temperature and humidity interactions.

In [1]:
# =========================
# 1. Master Imports
# =========================
import pandas as pd
import numpy as np
import altair as alt
from vega_datasets import data
import os

# =========================
# 2. Data Loading
# =========================
path = "/kaggle/input/datasets/xjoannax88/global-weather-and-air-pollution-dataset/output/openweather_weather_airpollution_top3cities_per_country.csv"
df = pd.read_csv(path)

display(df.head(3))

,collection_time_utc,source_weather,source_air,source_city_selection,units_weather,country_code,country_name,country_capital,country_population,continent,...,hour,day_of_week,is_weekend,season,temp_humidity_interaction,wind_pm25_interaction,pressure_temp_interaction,target_aqi_class,target_pm25,target_pm10
0,2026-04-24T07:06:06.738573+00:00,OpenWeather,OpenWeather,GeoNames cities15000 + countryInfo,metric,AD,Andorra,Andorra la Vella,77006,EU,...,7,4,0,spring,1058.25,2.1195,14392.20,2,4.71,10.81
1,2026-04-24T07:06:06.915310+00:00,OpenWeather,OpenWeather,GeoNames cities15000 + countryInfo,metric,AD,Andorra,Andorra la Vella,77006,EU,...,7,4,0,spring,1074.75,2.1195,14616.60,2,4.71,10.81
2,2026-04-24T07:06:07.088759+00:00,OpenWeather,OpenWeather,GeoNames cities15000 + countryInfo,metric,AE,United Arab Emirates,Abu Dhabi,9630959,AS,...,7,4,0,spring,860.16,87.7869,36162.56,4,28.41,116.85


# 2. Data Overview

In [2]:
print(f"Dataset Shape: {df.shape}\n")
df.info()
display(df.describe())
display(df.head())

Dataset Shape: (583, 88)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 583 entries, 0 to 582
Data columns (total 88 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   collection_time_utc         583 non-null    object 
 1   source_weather              583 non-null    object 
 2   source_air                  583 non-null    object 
 3   source_city_selection       583 non-null    object 
 4   units_weather               583 non-null    object 
 5   country_code                583 non-null    object 
 6   country_name                583 non-null    object 
 7   country_capital             583 non-null    object 
 8   country_population          583 non-null    int64  
 9   continent                   509 non-null    object 
 10  city_name                   583 non-null    object 
 11  city_name_ascii             583 non-null    object 
 12  city_population             583 non-null    int64  
 13  populatio

,country_population,city_population,population_rank_in_country,city_dem,geonameid,geo_lat,geo_lon,weather_id,temp,feels_like,...,day,hour,day_of_week,is_weekend,temp_humidity_interaction,wind_pm25_interaction,pressure_temp_interaction,target_aqi_class,target_pm25,target_pm10
count,5.830000e+02,5.830000e+02,583.000000,583.000000,5.830000e+02,583.000000,583.000000,583.000000,583.000000,583.000000,...,583.0,583.000000,583.0,583.0,583.000000,583.000000,583.000000,583.000000,583.000000,583.000000
mean,3.916960e+07,1.427771e+06,1.955403,259.061750,2.236879e+06,19.081813,19.660866,772.735849,20.661389,20.915986,...,24.0,6.972556,4.0,0.0,1342.856895,34.309148,20908.073156,1.936535,10.062556,22.225077
std,1.445742e+08,2.761411e+06,0.819480,1011.301057,1.867436e+06,23.906767,64.117286,94.975393,8.022589,9.599028,...,0.0,0.163514,0.0,0.0,638.551223,59.779243,8058.296641,0.969156,16.235549,42.711980
min,3.371800e+04,1.529800e+04,1.000000,-9999.000000,5.365400e+04,-43.533330,-175.201140,211.000000,-0.650000,-7.640000,...,24.0,6.000000,4.0,0.0,-39.000000,0.000000,-661.700000,1.000000,0.500000,0.500000
25%,2.108132e+06,1.135235e+05,1.000000,16.000000,9.101425e+05,4.954340,-9.310510,800.000000,14.235000,13.615000,...,24.0,7.000000,4.0,0.0,775.260000,3.636900,14493.205000,1.000000,1.435000,2.965000
50%,8.606316e+06,4.181400e+05,2.000000,58.000000,2.234974e+06,17.991070,21.011780,802.000000,21.270000,21.510000,...,24.0,7.000000,4.0,0.0,1316.250000,9.699600,21567.780000,2.000000,4.330000,7.670000
75%,2.849869e+07,1.448123e+06,3.000000,391.500000,3.192346e+06,40.286630,48.070555,804.000000,26.250000,26.365000,...,24.0,7.000000,4.0,0.0,1940.130000,34.790050,26538.750000,2.000000,11.625000,20.335000
max,1.411779e+09,2.487450e+07,3.000000,3782.000000,1.352732e+07,64.135480,178.513130,804.000000,41.580000,45.600000,...,24.0,7.000000,4.0,0.0,2595.000000,427.329000,41704.740000,5.000000,149.940000,369.080000


,collection_time_utc,source_weather,source_air,source_city_selection,units_weather,country_code,country_name,country_capital,country_population,continent,...,hour,day_of_week,is_weekend,season,temp_humidity_interaction,wind_pm25_interaction,pressure_temp_interaction,target_aqi_class,target_pm25,target_pm10
0,2026-04-24T07:06:06.738573+00:00,OpenWeather,OpenWeather,GeoNames cities15000 + countryInfo,metric,AD,Andorra,Andorra la Vella,77006,EU,...,7,4,0,spring,1058.25,2.1195,14392.20,2,4.71,10.81
1,2026-04-24T07:06:06.915310+00:00,OpenWeather,OpenWeather,GeoNames cities15000 + countryInfo,metric,AD,Andorra,Andorra la Vella,77006,EU,...,7,4,0,spring,1074.75,2.1195,14616.60,2,4.71,10.81
2,2026-04-24T07:06:07.088759+00:00,OpenWeather,OpenWeather,GeoNames cities15000 + countryInfo,metric,AE,United Arab Emirates,Abu Dhabi,9630959,AS,...,7,4,0,spring,860.16,87.7869,36162.56,4,28.41,116.85
3,2026-04-24T07:06:07.264082+00:00,OpenWeather,OpenWeather,GeoNames cities15000 + countryInfo,metric,AE,United Arab Emirates,Abu Dhabi,9630959,AS,...,7,4,0,spring,1254.76,65.0520,33350.20,3,18.07,97.68
4,2026-04-24T07:06:07.439562+00:00,OpenWeather,OpenWeather,GeoNames cities15000 + countryInfo,metric,AE,United Arab Emirates,Abu Dhabi,9630959,AS,...,7,4,0,spring,864.00,113.2200,36360.00,4,31.45,122.41


In [3]:
df.nunique()[df.nunique() < 10]

source_weather                1
source_air                    1
source_city_selection         1
units_weather                 1
continent                     5
population_rank_in_country    3
geonames_feature_code         6
weather_main                  9
rain_3h                       1
snow_1h                       1
snow_3h                       1
station_type                  1
cod                           1
is_daytime                    2
temp_category                 5
has_rain                      2
has_snow                      1
aqi                           5
aqi_label                     5
pm25_category                 4
year                          1
month                         1
day                           1
hour                          2
day_of_week                   1
is_weekend                    1
season                        1
target_aqi_class              5
dtype: int64

In [4]:
df.nunique()[df.nunique() > 100]

collection_time_utc           583
country_code                  209
country_name                  209
country_capital               209
country_population            209
city_name                     580
city_name_ascii               580
city_population               579
city_timezone_geonames        218
city_dem                      308
geonameid                     583
geonames_modification_date    196
geo_lat                       582
geo_lon                       583
temp                          525
feels_like                    525
temp_min                      519
temp_max                      521
grnd_level                    162
wind_speed                    243
wind_deg                      207
wind_gust                     238
weather_time_unix             200
weather_time_utc              200
sunrise_unix                  573
sunrise_utc                   573
sunset_unix                   578
sunset_utc                    578
station_country               209
city_id_openwe

# 3. Data Cleaning

In [5]:
# Filter all columns with one unique values
df = df.loc[:, df.nunique() > 1]

In [6]:
df[df.nunique()[df.nunique() < 10].index].apply(lambda x: x.unique())

continent                                             [EU, AS, nan, AF, SA, OC]
population_rank_in_country                                            [1, 2, 3]
geonames_feature_code                     [PPLC, PPLA, PPLA2, PPL, PPLG, PPLA3]
weather_main                  [Clouds, Clear, Rain, Drizzle, Haze, Fog, Thun...
is_daytime                                                               [1, 0]
temp_category                                 [mild, hot, warm, cold, freezing]
has_rain                                                                 [0, 1]
aqi                                                             [2, 4, 3, 1, 5]
aqi_label                               [fair, poor, moderate, good, very_poor]
pm25_category                                  [low, high, moderate, very_high]
hour                                                                     [7, 6]
target_aqi_class                                                [2, 4, 3, 1, 5]
dtype: object

In [7]:
missing_data = df.isnull().sum()
display(missing_data[missing_data > 0])

continent           74
wind_gust          281
station_country      1
city_name_api        1
dtype: int64

In [8]:
city_match = (df['city_name'] == df['city_name_ascii']).mean() * 100
print(f"City Name vs ASCII Match: {city_match:.2f}%")

if 'station_country' in df.columns:
    country_match = (df['country_code'] == df['station_country']).mean() * 100
    print(f"Country Code vs Station Country Match: {country_match:.2f}%")

City Name vs ASCII Match: 88.16%
Country Code vs Station Country Match: 99.83%


In [9]:
# Drop columns with missing values
cols_to_drop = ['wind_gust', 'continent', 'station_country', 'city_name_api']
df = df.drop(columns=cols_to_drop, errors='ignore')

In [10]:
# IDs & Duplicate Names
id_cols = ['city_name_ascii', 'geonameid', 'city_id_openweather'] 

# Duplicate Time Formats (Keeping parsed day/hour/month and the primary UTC datetime)
time_cols = ['weather_time_unix', 'weather_time_utc', 'sunrise_unix', 
             'sunrise_utc', 'sunset_unix', 'sunset_utc', 'air_time_unix', 'air_time_utc']

# Admin & API Metadata
admin_cols = ['collection_time_utc', 'geonames_modification_date', 'weather_icon']

# Data Leakage (Target Variables)
leakage_cols = ['aqi', 'aqi_label', 'target_pm25', 'target_pm10', 'pm2_5', 'pm10']

# Combine all lists and drop
all_useless_cols = id_cols + time_cols + admin_cols + leakage_cols

print(f"Dataframe shape before drop: {df.shape}")
df = df.drop(columns=all_useless_cols, errors='ignore')
print(f"Dataframe shape after drop: {df.shape}")

Dataframe shape before drop: (583, 68)
Dataframe shape after drop: (583, 48)


In [11]:
# Convert strings to Datetime objects
df['weather_datetime_utc'] = pd.to_datetime(df['weather_datetime_utc'])

# Convert text categories to actual Pandas Categories
cat_cols = ['temp_category', 'aqi_label', 'pm25_category', 'season']
for col in cat_cols:
    if col in df.columns:
        df[col] = df[col].astype('category')
        
# Final check of our memory usage and datatypes!
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 583 entries, 0 to 582
Data columns (total 48 columns):
 #   Column                      Non-Null Count  Dtype              
---  ------                      --------------  -----              
 0   country_code                583 non-null    object             
 1   country_name                583 non-null    object             
 2   country_capital             583 non-null    object             
 3   country_population          583 non-null    int64              
 4   city_name                   583 non-null    object             
 5   city_population             583 non-null    int64              
 6   population_rank_in_country  583 non-null    int64              
 7   city_timezone_geonames      583 non-null    object             
 8   geonames_feature_code       583 non-null    object             
 9   city_dem                    583 non-null    int64              
 10  geo_lat                     583 non-null    float64           

# 4. Exploratory Data Analysis (EDA)

In [12]:
outlier_cols = ['temp', 'humidity', 'wind_speed', 'co', 'no2', 'o3']

df_melted = df[outlier_cols].melt(var_name='Feature', value_name='Value')

outlier_plot = alt.Chart(df_melted).mark_boxplot(
    size=40,
    outliers=alt.MarkConfig(color='red', size=25) 
).encode(
    x=alt.X('Feature:N', title='', axis=alt.Axis(labels=False, ticks=False)),
    y=alt.Y('Value:Q', title='Measurement'),
    color=alt.Color('Feature:N', legend=None),
    tooltip=['Feature:N', 'Value:Q'] 
).properties(
    width=120,
    height=300
).facet(
    column=alt.Column('Feature:N', header=alt.Header(labelOrient='bottom', title='Weather & Air Quality Variables'))
).resolve_scale(
    y='independent' 
).configure_title(
    fontSize=16,
    anchor='middle'
)

outlier_plot

alt.FacetChart(...)

In [13]:
business_features = [
    'temp', 'humidity', 'wind_speed', 'rain_1h', 
    'temp_humidity_interaction', 'co', 'no2', 'o3', 'target_aqi_class'
]

corr_matrix = df[business_features].corr().reset_index().melt('index')
corr_matrix.columns = ['Variable 1', 'Variable 2', 'Correlation']

heatmap = alt.Chart(corr_matrix).mark_rect().encode(
    x=alt.X('Variable 1:O', title=''),
    y=alt.Y('Variable 2:O', title=''),
    color=alt.Color('Correlation:Q', scale=alt.Scale(scheme='redblue', domain=[-1, 1])),
    tooltip=[
        alt.Tooltip('Variable 1:O'),
        alt.Tooltip('Variable 2:O'),
        alt.Tooltip('Correlation:Q', format='.2f')
    ]
).properties(
    width=500,
    height=500,
    title="Business Driver Correlations: Weather vs. Air Quality"
)

text = heatmap.mark_text(baseline='middle').encode(
    text=alt.Text('Correlation:Q', format='.2f'),
    color=alt.condition(
        abs(alt.datum.Correlation) > 0.4, 
        alt.value('white'), 
        alt.value('black')
    )
)

heatmap + text

alt.LayerChart(...)

In [14]:
time_cols = ['hour', 'co', 'no2', 'o3']
df_hourly = df[time_cols].groupby('hour').mean().reset_index()

df_time_melted = df_hourly.melt('hour', var_name='Pollutant', value_name='Average Level')

time_chart = alt.Chart(df_time_melted).mark_line(
    point=alt.OverlayMarkDef(size=60),
    strokeWidth=3
).encode(
    x=alt.X('hour:O', title='Hour of the Day (UTC)'),
    y=alt.Y('Average Level:Q', title='Average Concentration'),
    color=alt.Color('Pollutant:N', title='Pollutant'),
    tooltip=[
        alt.Tooltip('hour:O', title='Hour'), 
        alt.Tooltip('Pollutant:N'), 
        alt.Tooltip('Average Level:Q', format='.2f')
    ]
).properties(
    width=600,
    height=400,
    title='Daily Pollution Rhythm: The Commuter & Sunlight Effect'
).resolve_scale(
    y='independent'
).configure_title(
    fontSize=18,
    anchor='middle'
)

time_chart

alt.Chart(...)

In [15]:
world = alt.topo_feature(data.world_110m.url, 'countries')

background = alt.Chart(world).mark_geoshape(
    fill='lightgray',
    stroke='white'
).project(
    type='naturalEarth1'
).properties(
    width=750,
    height=450,
    title='Global Market Targeting: Where are the Worst Pollution Hotspots?'
)

points = alt.Chart(df).mark_circle(opacity=0.7).encode(
    longitude='geo_lon:Q',
    latitude='geo_lat:Q',
    size=alt.Size('target_aqi_class:Q', scale=alt.Scale(range=[20, 400]), title='AQI Severity'),
    color=alt.Color('target_aqi_class:O', scale=alt.Scale(scheme='oranges'), title='AQI Class'),
    tooltip=[
        alt.Tooltip('city_name:N', title='City'), 
        alt.Tooltip('country_name:N', title='Country'), 
        alt.Tooltip('target_aqi_class:Q', title='AQI Class'),
        alt.Tooltip('temp:Q', title='Temperature (C)')
    ]
)

geo_map = (background + points).configure_title(
    fontSize=18,
    anchor='middle'
)

geo_map

alt.LayerChart(...)

# 5. Feature Engineering & Data Products
Transforming raw meteorological insights into actionable triggers for dynamic marketing, supply chain logistics, and healthcare alerts.

In [16]:
# Base Variables & Thresholds
df['heat_index'] = df['temp'] * df['humidity'] / 100
HEATWAVE = df['temp'].quantile(0.90)
HIGH_POLLUTION = 3

df['is_high_pollution'] = df['target_aqi_class'] >= HIGH_POLLUTION
df['is_heatwave'] = df['temp'] > HEATWAVE

# Marketing & Retail (Dynamic Ads)
df['marketing_signal'] = np.where((df['is_high_pollution']) | (df['is_heatwave']), 1, 0)
df['recommended_product'] = np.where(df['is_high_pollution'], "Air Purifier / Mask", 
                            np.where(df['is_heatwave'], "AC / Cooler", "None"))

# Supply Chain (Demand Signals)
df['umbrella_demand'] = (df['rain_1h'] > 0).astype(int)
df['winter_demand'] = (df['temp'] < 15).astype(int)
df['cooler_demand'] = (df['temp'] > 30).astype(int)

# Tourism & Events (Risk Score)
df['event_risk'] = ((df['rain_1h'] > 0) | (df['is_heatwave']) | (df['is_high_pollution'])).astype(int)

# Energy Grid (Load Proxy)
df['energy_load'] = (df['temp'] * 0.7 + df['humidity'] * 0.3)

# Gig Economy (Surge Pricing)
df['delivery_demand'] = df['event_risk'] 
df['surge_multiplier'] = 1 + (df['delivery_demand'] * 0.5)

# Healthcare (Risk Profiling)
df['health_risk'] = (df['target_aqi_class'] * 0.6 + df['heat_index'] * 0.4)
df['high_health_risk'] = df['health_risk'] > df['health_risk'].quantile(0.80)

# final business features for dashboarding
business_features = [
    'city_name', 'country_name', 'weather_datetime_utc', 'target_aqi_class', 
    'heat_index', 'marketing_signal', 'recommended_product', 'surge_multiplier', 
    'energy_load', 'high_health_risk'
]

final_products_df = df[business_features]
final_products_df.to_csv("business_features.csv", index=False)

display(final_products_df.head())

,city_name,country_name,weather_datetime_utc,target_aqi_class,heat_index,marketing_signal,recommended_product,surge_multiplier,energy_load,high_health_risk
0,Andorra la Vella,Andorra,2026-04-24 07:06:06+00:00,2,10.5825,0,None,1.0,32.377,False
1,les Escaldes,Andorra,2026-04-24 07:06:06+00:00,2,10.7475,0,None,1.0,32.531,False
2,Dubai,United Arab Emirates,2026-04-24 07:01:41+00:00,4,8.6016,1,Air Purifier / Mask,1.5,32.288,False
3,Abu Dhabi,United Arab Emirates,2026-04-24 07:06:07+00:00,3,12.5476,1,Air Purifier / Mask,1.5,34.514,False
4,Sharjah,United Arab Emirates,2026-04-24 07:06:07+00:00,4,8.6400,1,Air Purifier / Mask,1.5,32.400,False


# 6. Strategic Business Conclusions

Through our exploratory data analysis, we have transformed raw meteorological and atmospheric data into actionable commercial intelligence. By understanding the **what, why, when, and where** of air quality, businesses can transition from reactive operations to proactive strategies.

### 💡 Core Analytical Insights
1. **Geospatial Targeting (The "Where"):** Extreme pollution is heavily concentrated in equatorial regions across Asia and the Middle East. Global marketing budgets for health/HVAC products should be disproportionately allocated here.
2. **Temporal Volatility (The "When"):** Pollution is highly volatile (e.g., Ozone drops sharply without sunlight). Businesses cannot rely on daily averages; they need real-time, hour-by-hour API triggers.
3. **Meteorological Triggers (The "Why"):** Air quality does not exist in a vacuum. Variables like the **Temperature-Humidity Interaction** act as massive leading indicators for hazardous air days.

### 🛠️ The Deliverable: A Dashboard-Ready Data Product
Instead of purely descriptive analytics, Section 5 of this notebook engineered a final, deployable data product (`business_features.csv`). We successfully translated raw weather metrics into automated, rule-based business signals, including:
* A dynamic **Surge Multiplier** for gig-economy logistics.
* A Boolean **Marketing Signal** to trigger automated ad spend.
* A composite **Energy Load** proxy and **Health Risk** index.

### 🚀 Future Scope: Predictive Machine Learning
While this notebook successfully builds rule-based triggers, the ultimate evolution of this product is **Predictive Analytics**. By feeding our cleaned, highly-correlated feature set into a Machine Learning algorithm (like XGBoost or Random Forest), companies could move from reacting to *current* conditions to automating their pricing and logistics based purely on *tomorrow's* forecast.